# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading and exploring the FAIR^2 Croissant-formatted dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata is an object; use the .to_json() method to inspect fields
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. This step is important for understanding the available data structure and for referencing correct IDs in downstream steps.

In [ ]:
# List all available record sets and their field IDs
print("Available record set @ids and field @ids:")

# Accessing .record_sets returns a list of RecordSet objects
if not dataset.record_sets:
    print('No record sets found in the dataset schema.')
else:
    for recset in dataset.record_sets:
        print(f"- Record Set @id: {recset.id}")
        print(f"  Name: {getattr(recset, 'name', '')}")
        print(f"  Fields:")
        for field in recset.fields:
            print(f"    - Field @id: {field.id} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")
        print()

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for further analysis. All entities (record sets, fields, and columns) are referenced by their `@id` fields as per best practice.

> **Tip**: Use the printed list of record set and field `@id`s from the previous cell to select the appropriate data for your analysis.

In [ ]:
# Collect all record set @ids
record_set_ids = [recset.id for recset in dataset.record_sets]
dataframes = {}
for recset_id in record_set_ids:
    # Load all records for this record set
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[recset_id] = df
            print(f"Loaded DataFrame for Record Set @id: {recset_id}; shape: {df.shape}")
        else:
            print(f"No records found for Record Set @id: {recset_id}")
    except Exception as e:
        print(f"Error loading records for Record Set @id: {recset_id}: {e}")

# Show column names for one DataFrame if available:
if dataframes:
    # Display columns of the first loaded DataFrame
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nAvailable columns in DataFrame for Record Set @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    print("\nPreview of the DataFrame:")
    display(dataframes[first_rs_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing and analysis steps. This may include filtering records, normalizing numeric fields, and grouping by attributes for aggregation. 
All fields are referenced by their `@id` for maximum clarity and reproducibility.

In [ ]:
# For demonstration, select the first record set and look for numeric fields
import numpy as np
if dataframes:
    record_set_id = first_rs_id  # Just for this demonstration
    df = dataframes[record_set_id]

    # Try to infer numeric fields by datatype or value
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    print(f"Numeric fields in the selected record set: {numeric_fields}")

    # If we find a numeric field, perform filtering and normalization
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use the first numeric field
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field (not numeric, not index)
        candidate_group_fields = [col for col in df.columns if col not in numeric_fields]
        group_field = candidate_group_fields[0] if candidate_group_fields else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric fields detected for EDA demonstration.")
else:
    print("No dataframes available to perform EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields. Let's plot a histogram of a chosen numeric field and a boxplot grouped by a categorical field (if detected).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if EDA could identify fields
if dataframes and 'numeric_field' in locals() and 'group_field' in locals() and numeric_field and group_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if df[group_field].nunique() < 20:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: no numeric and group field detected or no data available.')

## 6. Conclusion

In this notebook, we demonstrated loading and exploring a Croissant-structured dataset using `mlcroissant`, with explicit reference to all entities by their `@id` fields. We loaded the dataset, reviewed record set and field IDs, extracted DataFrames for analysis, performed preliminary exploratory data analysis, and visualized the results. This reproducible approach enables easy integration of FAIR and machine-actionable datasets in Python workflows.